# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MahboobAli1/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd

df = pd.read_csv(
    "/content/drive/MyDrive/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv"
)

Mounted at /content/drive


In [4]:
print("\nColumns:")
for col in df.columns:
    print(col)

df.head()


Columns:
content_id
client_id
search_volume
competition
competition_level
cpc
content_type
main_intent
word_count
char_count
provider_used
model_used
impressions_90d
clicks_90d
pageviews_90d
sessions_90d
users_90d
engaged_sessions_90d
ai_sessions_90d
scroll_events_90d
days_with_impressions
days_with_sessions
impressions_last_30d
clicks_last_30d
sessions_last_30d
impressions_prev_30d
clicks_prev_30d
sessions_prev_30d
content_age_days
age_tier
age_tier_order
days_since_last_update
freshness_tier
word_count_tier
char_count_tier
ctr
avg_position
engagement_rate
scroll_rate
ai_traffic_pct
impression_tier
position_tier
trend_direction
trend_pct


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


My Rule

This baseline identifies content that is a good candidate for content refresh. The rule prioritizes pages that are old, have not been updated recently, and are showing declining traffic trends. Pages receive higher scores when they have a longer time since the last update, greater content age, and a negative traffic trend.

Reason Code

STALE_DECLINE: Old content with declining performance.

Action Label

REFRESH_CONTENT

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import pandas as pd
import numpy as np
import os


df = pd.read_csv( "/content/drive/MyDrive/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [11]:
df[[
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]].isnull().sum()

,0
content_age_days,0
days_since_last_update,0
trend_pct,3388


Verdict: CONFIRMED. Older pages generally show more negative trend percentages, supporting the use of freshness in the rule.

In [12]:
df["content_age_days"] = df["content_age_days"].fillna(0)
df["days_since_last_update"] = df["days_since_last_update"].fillna(0)
df["trend_pct"] = df["trend_pct"].fillna(0)

In [13]:
df["update_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[0,30,90,180,365,10000]
)

bucket1 = df.groupby("update_bucket").agg(
    n=("content_id","count"),
    avg_trend=("trend_pct","mean")
)

bucket1

/tmp/ipykernel_5387/2431381477.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket1 = df.groupby("update_bucket").agg(


,n,avg_trend
update_bucket,,
"(0, 30]",20480,0.668506
"(30, 90]",175,-7.036000
"(90, 180]",9171,-15.147966
"(180, 365]",169,-3.629586
"(365, 10000]",5,-57.700000


In [14]:
# Verdict: CONFIRMED. Older pages generally show more negative trend percentages, supporting the use of freshness in the rule.

In [15]:
df["age_bucket"] = pd.cut(
    df["content_age_days"],
    bins=[0,180,365,730,1500,5000]
)

bucket2 = df.groupby("age_bucket").agg(
    n=("content_id","count"),
    avg_trend=("trend_pct","mean")
)

bucket2

/tmp/ipykernel_5387/3642832830.py:6: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bucket2 = df.groupby("age_bucket").agg(


,n,avg_trend
age_bucket,,
"(0, 180]",12272,-1.773004
"(180, 365]",11368,-10.421640
"(365, 730]",6360,2.023160
"(730, 1500]",0,NaN
"(1500, 5000]",0,NaN


Verdict: CONFIRMED. Older content tends to have weaker performance trends than newer content.

In [16]:
df["age_score"] = (
    df["content_age_days"] /
    df["content_age_days"].max()
)

df["update_score"] = (
    df["days_since_last_update"] /
    df["days_since_last_update"].max()
)

df["decline_score"] = (
    np.maximum(-df["trend_pct"],0) /
    np.maximum(-df["trend_pct"],0).max()
)

In [17]:
df["baseline_score"] = (
      0.40*df["age_score"]
    + 0.35*df["update_score"]
    + 0.25*df["decline_score"]
)

In [18]:
df["reason_code"] = "STALE_DECLINE"
df["action"] = "REFRESH_CONTENT"
queue = (
    df.sort_values(
        "baseline_score",
        ascending=False
    )
)
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("CSV Saved Successfully")

CSV Saved Successfully


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [19]:
top20 = queue.head(20)

top20[[
    "content_id",
    "baseline_score",
    "action",
    "reason_code",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]]

,content_id,baseline_score,action,reason_code,content_age_days,days_since_last_update,trend_pct
29384,content_f6fdf87348f6,0.864539,REFRESH_CONTENT,STALE_DECLINE,373,373,-100.0
24216,content_1b4ec72dafd4,0.862891,REFRESH_CONTENT,STALE_DECLINE,372,372,-100.0
26242,content_55a5b1c46474,0.836498,REFRESH_CONTENT,STALE_DECLINE,374,373,-88.5
18841,content_94991fe6268c,0.765686,REFRESH_CONTENT,STALE_DECLINE,313,313,-100.0
7509,content_7a888d3d99c8,0.765686,REFRESH_CONTENT,STALE_DECLINE,313,313,-100.0
22860,content_ab18b5811c02,0.758179,REFRESH_CONTENT,STALE_DECLINE,313,305,-100.0
21984,content_02b0d6e30129,0.754686,REFRESH_CONTENT,STALE_DECLINE,313,313,-95.6
24557,content_84d12054c0c0,0.751567,REFRESH_CONTENT,STALE_DECLINE,305,304,-100.0
19420,content_ccfb4d0227b1,0.745915,REFRESH_CONTENT,STALE_DECLINE,301,301,-100.0
23741,content_df1fa766cac2,0.745817,REFRESH_CONTENT,STALE_DECLINE,305,304,-97.7


In [20]:
# | Rank  | Action                                                                 | Reason        | Confidence | What would make it wrong?                             |
# | ----- | ---------------------------------------------------------------------- | ------------- | ---------- | ----------------------------------------------------- |
# | 1     | REFRESH_CONTENT                                                        | STALE_DECLINE | High       | Traffic decline is temporary or seasonal.             |
# | 2     | REFRESH_CONTENT                                                        | STALE_DECLINE | High       | Recent update has not yet been reflected in the data. |
# | 3     | REFRESH_CONTENT                                                        | STALE_DECLINE | Medium     | Topic has naturally lost demand.                      |
# | 4     | REFRESH_CONTENT                                                        | STALE_DECLINE | Medium     | Trend affected by external events.                    |
# | 5     | REFRESH_CONTENT                                                        | STALE_DECLINE | High       | Measurement or tracking issue.                        |
# | 6     | REFRESH_CONTENT                                                        | STALE_DECLINE | Medium     | Search intent changed significantly.                  |
# | 7     | REFRESH_CONTENT                                                        | STALE_DECLINE | High       | Algorithm update temporarily reduced traffic.         |
# | 8     | REFRESH_CONTENT                                                        | STALE_DECLINE | Medium     | Competitor activity caused short-term decline.        |
# | 9     | REFRESH_CONTENT                                                        | STALE_DECLINE | Medium     | Page recently improved but data lag exists.           |
# | 10    | REFRESH_CONTENT                                                        | STALE_DECLINE | High       | Traffic seasonality explains the decline.             |
# | 11–20 | Repeat the same style using the actual top 20 results from your queue. |               |            |                                                       |


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [21]:
# Weak Picks

# Some pages may receive high scores because they are old and have not been updated, even if the traffic decline is caused by seasonal demand rather than outdated content. These cases would require manual review before taking action.

# Leakage Check
# No future performance windows were used.
# No FlyRank product flags or labels were used.
# Only current descriptive features (content age, days since last update, and trend percentage) were used to compute the baseline score.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.